# On the Number of Eigenvectors

*Course notes for **Math for Machine Learning**, C1 · W4 · L2 · V05b — "On the Number of Eigenvectors" (DeepLearning.AI).*

The last $3\times 3$ example had three distinct eigenvalues and three eigenvectors. **Does every $3\times 3$ matrix have three eigenvectors?** No! When an eigenvalue is **repeated**, the number of independent eigenvectors can fall short. We explore two matrices that differ by a **single entry** yet behave very differently, then summarize the general rules.

In [1]:
import numpy as np
from scipy.linalg import null_space

np.set_printoptions(precision=3, suppress=True)

## 1. A repeated eigenvalue that still gives enough eigenvectors

$$ A_1 = \begin{pmatrix} 2 & 0 & 0 \\ -1 & 4 & -0.5 \\ 0 & 0 & 2 \end{pmatrix}. $$

Its characteristic polynomial is $(2-\lambda)^2(4-\lambda)$, so the eigenvalues are $4, 2, 2$ — **$2$ is repeated**. For $\lambda = 4$ we get the eigenvector $(0,1,0)$. For $\lambda = 2$ the single equation $-x_1 + 2x_2 - 0.5x_3 = 0$ leaves **two free variables** — a whole **2D plane** of eigenvectors, e.g. $(2,1,0)$ and $(1,1,2)$.

So even with a repeated eigenvalue, $A_1$ has **three** independent eigenvectors — a full eigenbasis.

In [2]:
A1 = np.array([[ 2,  0,  0],
               [-1,  4, -0.5],
               [ 0,  0,  2]])
print('char poly coeffs =', np.poly(A1))
print('eigenvalues      =', np.sort(np.roots(np.poly(A1))), ' -> 2 is repeated')
print()
for lam in [4, 2]:
    ns = null_space(A1 - lam * np.eye(3))
    print(f'lambda = {lam}: eigenspace dimension = {ns.shape[1]}')
    print(np.round(ns, 3))
    print()
print('total independent eigenvectors:', sum(null_space(A1 - l*np.eye(3)).shape[1] for l in [4, 2]))

char poly coeffs = [  1.  -8.  20. -16.]
eigenvalues      = [2. 2. 4.]  -> 2 is repeated

lambda = 4: eigenspace dimension = 1
[[ 0.]
 [-1.]
 [ 0.]]

lambda = 2: eigenspace dimension = 2
[[ 0.    -0.9  ]
 [ 0.243 -0.423]
 [ 0.97   0.106]]

total independent eigenvectors: 3


## 2. Change one entry — and lose an eigenvector

Change a single value (the bottom-left entry $0 \to 4$):

$$ A_2 = \begin{pmatrix} 2 & 0 & 0 \\ -1 & 4 & -0.5 \\ 4 & 0 & 2 \end{pmatrix}. $$

The characteristic polynomial is **unchanged** — eigenvalues are still $4, 2, 2$. But now for $\lambda = 2$ the equations force $x_1 = 0$ and $x_3 = 4x_2$, leaving just **one** free variable — a single direction $(0, 1, 4)$. Choosing another value of $x_2$ (say $\tfrac12$) gives $(0, \tfrac12, 2)$, which is just a **scaling** of the same vector — the *same* eigenvector.

So the repeated eigenvalue $2$ has **only one** eigenvector. $A_2$ has just **two** independent eigenvectors — **not enough** to form an eigenbasis of 3D space.

In [3]:
A2 = np.array([[ 2,  0,  0],
               [-1,  4, -0.5],
               [ 4,  0,  2]])   # only the (3,1) entry changed: 0 -> 4
print('char poly coeffs =', np.poly(A2), ' (same as A1)')
print('eigenvalues      =', np.sort(np.roots(np.poly(A2))))
print()
for lam in [4, 2]:
    ns = null_space(A2 - lam * np.eye(3))
    print(f'lambda = {lam}: eigenspace dimension = {ns.shape[1]}')
    print(np.round(ns, 3))
    print()
total = sum(null_space(A2 - l*np.eye(3)).shape[1] for l in [4, 2])
print('total independent eigenvectors:', total, '-> only', total, '< 3, no eigenbasis')

char poly coeffs = [  1.  -8.  20. -16.]  (same as A1)
eigenvalues      = [2. 2. 4.]

lambda = 4: eigenspace dimension = 1
[[0.]
 [1.]
 [0.]]

lambda = 2: eigenspace dimension = 1
[[-0.   ]
 [-0.243]
 [-0.97 ]]

total independent eigenvectors: 2 -> only 2 < 3, no eigenbasis


## 3. Algebraic vs. geometric multiplicity

The two notions in play:

- **Algebraic multiplicity** — how many times $\lambda$ is a **root** of the characteristic polynomial.
- **Geometric multiplicity** — how many **independent eigenvectors** $\lambda$ actually has (the dimension of its eigenspace, i.e. the null space of $A - \lambda I$).

Geometric multiplicity is **at most** the algebraic one. When it's strictly smaller (as for $A_2$'s $\lambda=2$: algebraic $2$, geometric $1$), the matrix is **defective** — it lacks a full set of eigenvectors and has **no eigenbasis**.

In [4]:
def multiplicities(A):
    vals = np.roots(np.poly(A))
    uniq = np.unique(np.round(vals, 6))
    for lam in uniq:
        alg = int(np.sum(np.isclose(vals, lam)))
        geo = null_space(A - lam * np.eye(A.shape[0])).shape[1]
        flag = '' if alg == geo else '   <-- defective (geo < alg)'
        print(f'lambda = {lam:>5.1f}:  algebraic = {alg}, geometric = {geo}{flag}')

print('A1:'); multiplicities(A1)
print('\nA2:'); multiplicities(A2)

A1:
lambda =   2.0:  algebraic = 2, geometric = 2
lambda =   4.0:  algebraic = 1, geometric = 1

A2:
lambda =   2.0:  algebraic = 2, geometric = 1   <-- defective (geo < alg)
lambda =   4.0:  algebraic = 1, geometric = 1


## 4. The general rules

How many independent eigenvectors can you get?

**$2\times 2$ matrix** (eigenvalues $\lambda_1, \lambda_2$):

| case | # eigenvectors |
|---|---|
| $\lambda_1 \neq \lambda_2$ | always **2** |
| $\lambda_1 = \lambda_2$ | **1 or 2** |

**$3\times 3$ matrix** (eigenvalues $\lambda_1, \lambda_2, \lambda_3$):

| case | # eigenvectors |
|---|---|
| all three different | always **3** |
| one repeated twice, one distinct | **2 or 3** (our $A_2$ / $A_1$) |
| same value three times | anywhere from **1 to 3** |

**Distinct eigenvalues always give distinct eigenvectors**; only **repeated** eigenvalues create the uncertainty.

## Summary

- A repeated eigenvalue **may or may not** supply enough eigenvectors — two matrices differing by one entry can have $3$ vs $2$ independent eigenvectors.
- **Algebraic multiplicity** (root count) $\geq$ **geometric multiplicity** (independent eigenvectors $=$ $\dim\ker(A-\lambda I)$). When geometric $<$ algebraic, the matrix is **defective** and has **no eigenbasis**.
- **Distinct** eigenvalues always yield **distinct** eigenvectors; **repeated** eigenvalues are where the count can drop.
- $n \times n$ rule of thumb: all-distinct eigenvalues $\Rightarrow n$ eigenvectors (guaranteed eigenbasis); repetitions $\Rightarrow$ somewhere between $1$ and $n$.
- Count independent eigenvectors with `scipy.linalg.null_space(A - lam*I).shape[1]`.